# 5. cVAE + Adversarial Batch Correction (no KL)**Архитектура:**```Encoder (batch-FREE):  4224 → 512 → 256 → (μ, logσ²) → zDecoder (batch-COND):  (z + batch_onehot) → 256 → 512 → 4224Discriminator (GRL):   GRL(z) → 128 → n_batchesLoss:  MSE_recon + λ_adv × CE(batch_pred, batch_true)```**Отличие от cVAE (notebook 4):**- Нет KL дивергенции — она уничтожала биологию- Вместо KL — adversarial batch discriminator с Gradient Reversal Layer- GRL инвертирует градиент → encoder учится НЕ кодировать батч в латенте**Оценка:**- KIRC, NSCLC: Silhouette по batch и diagnosis- Immotion (ICI/TKI): Cohort как контроль пере-коррекции (1 лаборатория → нет батча)- Mariathasan (BLCA): Пермутационный тест (рандомные batch labels)

## 0. Environment Setup

In [ ]:
!git clone -b feat/scvi-batch-correction https://github.com/onion-42/batchcor-rna-embeds.git 2>/dev/null || (cd batchcor-rna-embeds && git pull origin feat/scvi-batch-correction)%cd batchcor-rna-embeds!pip install -q uv!uv pip install --system -e .!pip install -q --upgrade ipython ipykernel

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport torchfrom pathlib import Pathfrom loguru import loggerfrom matplotlib.lines import Line2Dfrom sklearn.metrics import silhouette_scorefrom umap import UMAPfrom batchcor_rna_emb.logging_config import set_loggerfrom batchcor_rna_emb.config import (    BATCH_COL, DIAGNOSIS_COL, SEED,    COMPASS_PT_EMBEDDING_KEY,    SCVI_CORRECTED_KEY,    COHORT_CANCER_CODE,)from batchcor_rna_emb.data_io import load_cohort, save_adata_zarrfrom batchcor_rna_emb.split_utils import add_dummy_split, get_split_masksfrom batchcor_rna_emb.batch_correction.scvi_corrector import CVAEAdvCorrector, CVAEAdvConfigset_logger(level="INFO")DEVICE = "cuda" if torch.cuda.is_available() else "cpu"logger.info("Device: {}, PyTorch: {}", DEVICE, torch.__version__)

In [ ]:
from google.colab import drivedrive.mount('/content/drive')DATA_INTERIM_PT_DIR = Path('/content/drive/MyDrive/data/interim_PT')DATA_PROCESSED_DIR  = Path('/content/drive/MyDrive/data/processed')       # old cVAE resultsDATA_PROCESSED_ADV  = Path('/content/drive/MyDrive/data/processed_adv')   # adversarial resultsDATA_PROCESSED_ADV.mkdir(parents=True, exist_ok=True)SPLIT_COL = "Split_dummy"ADV_KEY = SCVI_CORRECTED_KEY + "_adv"logger.info("Source: {}", [p.name for p in sorted(DATA_INTERIM_PT_DIR.glob('*.adata.zarr'))])

## 1. Helpers

In [ ]:
sns.set_theme(style="whitegrid", font_scale=1.0)plt.rcParams.update({"figure.facecolor": "#f8f9fa", "axes.facecolor": "#ffffff", "figure.dpi": 120})def plot_umap_panel(coords, labels, title, ax, palette=None):    unique = sorted(labels.unique())    if palette is None:        palette = sns.color_palette("husl", n_colors=len(unique))    cmap = dict(zip(unique, palette))    colors = [cmap[l] for l in labels]    ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=10, alpha=0.6, edgecolors="none", rasterized=True)    ax.set_title(title, fontsize=9, fontweight="bold")    if len(unique) <= 12:        handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=cmap[l], markersize=6, label=str(l)) for l in unique]        ax.legend(handles=handles, fontsize=5, loc="best", framealpha=0.8, edgecolor="#ccc")

## 2. Adversarial cVAE — обработка всех когортДля каждой когорты:1. Загрузка из `interim_PT`2. Fit Adv-cVAE на train → transform all3. Сохранение в `processed_adv/`

In [ ]:
adv_config = CVAEAdvConfig(    latent_dim=64,    hidden_dims=(512, 256),    n_epochs=150,    batch_size=128,    lr=1e-3,    lambda_adv=1.0,    warmup_fraction=0.3,    dropout=0.2,    normalize=True,    seed=SEED,)all_histories = {}for cohort_name in COHORT_CANCER_CODE:    src_path  = DATA_INTERIM_PT_DIR / f"{cohort_name}.adata.zarr"    dest_path = DATA_PROCESSED_ADV  / f"{cohort_name}.adata.zarr"    if not src_path.exists():        logger.warning("Missing: {}", cohort_name)        continue    if dest_path.exists():        logger.info("Already done, skipping: {}", cohort_name)        continue    logger.info("\n" + "=" * 70)    logger.info("Cohort: {}", cohort_name)    adata = load_cohort(src_path)    if COMPASS_PT_EMBEDDING_KEY not in adata.obsm:        logger.error("Missing embedding, skipping")        del adata; continue    adata = add_dummy_split(adata, col_name=SPLIT_COL, seed=SEED)    train_mask, test_mask = get_split_masks(adata, SPLIT_COL)    emb_all = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)    batch_labels = adata.obs[BATCH_COL].astype(str).values    n_unique_batches = len(set(batch_labels))    logger.info("Samples: {}, Batches: {}, Train: {}", adata.n_obs, n_unique_batches, train_mask.sum())    if n_unique_batches < 2:        logger.info("Single batch — saving raw embeddings as-is (no correction needed)")        adata.obsm[ADV_KEY] = emb_all    else:        corrector = CVAEAdvCorrector(config=adv_config)        corrector.fit(X=emb_all[train_mask], batch_labels=batch_labels[train_mask])        corrected = corrector.transform(emb_all)        adata.obsm[ADV_KEY] = corrected        all_histories[cohort_name] = corrector.history_        logger.info("Stored '{}' shape={}", ADV_KEY, corrected.shape)        assert np.isfinite(corrected).all()    save_adata_zarr(adata, dest_path)    logger.info("Saved: {}", dest_path)    del adata    torch.cuda.empty_cache() if DEVICE == "cuda" else Nonelogger.info("\nAll cohorts processed.")

## 3. Training Curves (Recon + Adversarial + Disc Accuracy)

In [ ]:
if not all_histories:    logger.warning("No histories to plot.")else:    for cname, hist in all_histories.items():        fig, axes = plt.subplots(1, 4, figsize=(22, 4))        fig.suptitle(f"{cname} — Adversarial cVAE Training", fontsize=13, fontweight="bold", y=1.03)        epochs = range(1, len(hist.total) + 1)        axes[0].plot(epochs, hist.recon, color='#2196F3', lw=1.5)        axes[0].set_title("Reconstruction (MSE)")        axes[1].plot(epochs, hist.adv, color='#FF5722', lw=1.5)        axes[1].set_title("Adversarial Loss (CE)")        axes[2].plot(epochs, hist.disc_accuracy, color='#9C27B0', lw=1.5)        axes[2].set_title("Discriminator Accuracy")        axes[2].axhline(y=1.0/len(set(batch_labels)), color='gray', ls='--', lw=1, label='chance')        axes[2].set_ylim(0, 1)        axes[2].legend(fontsize=8)        axes[3].plot(epochs, hist.lambda_schedule, color='#4CAF50', lw=1.5)        axes[3].set_title("λ_adv Schedule")        for ax in axes:            ax.set_xlabel("Epoch")        plt.tight_layout()        plt.show()

## 4. UMAP — Before vs After

In [ ]:
test_cohort = Nonefor name in COHORT_CANCER_CODE:    p = DATA_PROCESSED_ADV / f"{name}.adata.zarr"    if p.exists():        ad_tmp = load_cohort(p)        if ad_tmp.obs[BATCH_COL].nunique() > 1:            test_cohort = name            del ad_tmp; break        del ad_tmpif test_cohort:    adata = load_cohort(DATA_PROCESSED_ADV / f"{test_cohort}.adata.zarr")    batch = adata.obs[BATCH_COL].astype(str)    diag  = adata.obs[DIAGNOSIS_COL].astype(str)    batch_pal = sns.color_palette("husl", n_colors=batch.nunique())    diag_pal  = sns.color_palette("Set2", n_colors=diag.nunique())    emb_raw  = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)    emb_corr = np.asarray(adata.obsm[ADV_KEY], dtype=np.float32)    umap_raw  = UMAP(n_components=2, random_state=SEED).fit_transform(emb_raw)    umap_corr = UMAP(n_components=2, random_state=SEED).fit_transform(emb_corr)    fig, axes = plt.subplots(2, 2, figsize=(14, 12))    fig.suptitle(f"{test_cohort} — Before vs After Adv-cVAE", fontsize=14, fontweight="bold", y=1.02)    plot_umap_panel(umap_raw,  batch, "Before — Batch",     axes[0, 0], batch_pal)    plot_umap_panel(umap_corr, batch, "After — Batch",      axes[0, 1], batch_pal)    plot_umap_panel(umap_raw,  diag,  "Before — Diagnosis", axes[1, 0], diag_pal)    plot_umap_panel(umap_corr, diag,  "After — Diagnosis",  axes[1, 1], diag_pal)    for ax in axes.flat:        ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")    plt.tight_layout(); plt.show()    del adataelse:    logger.warning("No multi-batch cohort found.")

## 5. Silhouette Scores

In [ ]:
rows = []for cohort_name in COHORT_CANCER_CODE:    p = DATA_PROCESSED_ADV / f"{cohort_name}.adata.zarr"    if not p.exists(): continue    adata = load_cohort(p)    emb_raw  = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)    emb_corr = np.asarray(adata.obsm[ADV_KEY], dtype=np.float32)    batch = adata.obs[BATCH_COL].astype(str).values    diag  = adata.obs[DIAGNOSIS_COL].astype(str).values    mb = len(set(batch)) > 1; md_ = len(set(diag)) > 1    rows.append({        "Cohort": cohort_name,        "Sil_Batch_Before": round(silhouette_score(emb_raw, batch), 4) if mb else np.nan,        "Sil_Batch_After":  round(silhouette_score(emb_corr, batch), 4) if mb else np.nan,        "Sil_Diag_Before":  round(silhouette_score(emb_raw, diag), 4) if md_ else np.nan,        "Sil_Diag_After":   round(silhouette_score(emb_corr, diag), 4) if md_ else np.nan,    })    del adatadf_sil = pd.DataFrame(rows)df_sil["Sil_Batch_Δ"] = df_sil["Sil_Batch_After"] - df_sil["Sil_Batch_Before"]df_sil["Sil_Diag_Δ"]  = df_sil["Sil_Diag_After"]  - df_sil["Sil_Diag_Before"]display(df_sil)print("\nSil_Batch_Δ < 0 = лучшее смешивание батчей (хорошо)")print("Sil_Diag_Δ >= 0 = биология сохранена (хорошо)")

## 6. Immotion: контроль пере-коррекцииccRCC_ICI и ccRCC_TKI — два клинических трайла, но **одна лаборатория**.Используем `Cohort` как метрику: silhouette по Cohort НЕ должен меняться,потому что между трайлами нет реального батч-эффекта.

In [ ]:
immotion_cohorts = [    "PUB_ccRCC_Immotion150_and_151_ICI",    "PUB_ccRCC_Immotion150_and_151_TKI",]for cname in immotion_cohorts:    p = DATA_PROCESSED_ADV / f"{cname}.adata.zarr"    if not p.exists(): continue    adata = load_cohort(p)    emb_raw  = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)    emb_corr = np.asarray(adata.obsm[ADV_KEY], dtype=np.float32)    cohort_labels = adata.obs["Cohort"].astype(str).values    if len(set(cohort_labels)) > 1:        sil_raw  = silhouette_score(emb_raw, cohort_labels)        sil_corr = silhouette_score(emb_corr, cohort_labels)        print(f"{cname}:")        print(f"  Sil_Cohort Before: {sil_raw:.4f}")        print(f"  Sil_Cohort After:  {sil_corr:.4f}")        print(f"  Δ: {sil_corr - sil_raw:.4f}  (должен быть ~0)")    else:        print(f"{cname}: single Cohort value — skipped")    del adata

## 7. Mariathasan: пермутационный тестBLCA — один батч. Присваиваем рандомные batch labels (из распределенияRNA_batch других когорт), обучаем Adv-cVAE, замеряем silhouette.Если метод работает правильно, разница с некорректированными данными ~0.

In [ ]:
blca_name = "PUB_BLCA_Mariathasan_EGAS00001002556"blca_path = DATA_PROCESSED_ADV / f"{blca_name}.adata.zarr"if blca_path.exists():    adata = load_cohort(blca_path)    emb = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)    n = adata.n_obs    # Собираем пул batch labels из других когорт    batch_pool = []    for cname in COHORT_CANCER_CODE:        if cname == blca_name: continue        p = DATA_INTERIM_PT_DIR / f"{cname}.adata.zarr"        if not p.exists(): continue        ad_tmp = load_cohort(p)        batch_pool.extend(ad_tmp.obs[BATCH_COL].astype(str).values.tolist())        del ad_tmp    batch_pool = np.array(batch_pool)    logger.info("Batch pool size: {}, unique: {}", len(batch_pool), len(set(batch_pool)))    N_PERMS = 5    perm_results = []    for i in range(N_PERMS):        rng = np.random.RandomState(SEED + i)        fake_labels = rng.choice(batch_pool, size=n, replace=True)        n_fake = len(set(fake_labels))        if n_fake < 2:            logger.warning("Perm {}: only 1 unique label, skipping", i)            continue        cfg = CVAEAdvConfig(latent_dim=64, n_epochs=100, normalize=True, lambda_adv=1.0, seed=SEED+i)        corr = CVAEAdvCorrector(config=cfg)        corr.fit(X=emb, batch_labels=fake_labels)        emb_perm = corr.transform(emb)        sil_raw  = silhouette_score(emb, fake_labels)        sil_perm = silhouette_score(emb_perm, fake_labels)        perm_results.append({"perm": i, "n_fake_batches": n_fake,                             "sil_before": round(sil_raw, 4), "sil_after": round(sil_perm, 4),                             "delta": round(sil_perm - sil_raw, 4)})        logger.info("Perm {}: sil_before={:.4f}, sil_after={:.4f}, Δ={:.4f}", i, sil_raw, sil_perm, sil_perm - sil_raw)    if perm_results:        df_perm = pd.DataFrame(perm_results)        display(df_perm)        print(f"\nСредний Δ: {df_perm['delta'].mean():.4f} (должен быть ~0)")    del adataelse:    logger.warning("BLCA not found in processed_adv/")

## 8. Сравнение: старый cVAE (KL) vs Adversarial cVAEЕсли доступны результаты из `data/processed/` (notebook 4), сравниваем.

In [ ]:
rows_cmp = []for cohort_name in COHORT_CANCER_CODE:    old_p = DATA_PROCESSED_DIR / f"{cohort_name}.adata.zarr"    new_p = DATA_PROCESSED_ADV / f"{cohort_name}.adata.zarr"    if not old_p.exists() or not new_p.exists(): continue    ad_old = load_cohort(old_p)    ad_new = load_cohort(new_p)    batch = ad_old.obs[BATCH_COL].astype(str).values    diag  = ad_old.obs[DIAGNOSIS_COL].astype(str).values    mb = len(set(batch)) > 1; md_ = len(set(diag)) > 1    emb_raw = np.asarray(ad_old.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)    sil_b_raw = silhouette_score(emb_raw, batch) if mb else np.nan    sil_d_raw = silhouette_score(emb_raw, diag) if md_ else np.nan    # Old cVAE (KL)    if SCVI_CORRECTED_KEY in ad_old.obsm:        emb_old = np.asarray(ad_old.obsm[SCVI_CORRECTED_KEY], dtype=np.float32)        sil_b_old = silhouette_score(emb_old, batch) if mb else np.nan        sil_d_old = silhouette_score(emb_old, diag) if md_ else np.nan    else:        sil_b_old = sil_d_old = np.nan    # Adversarial cVAE    if ADV_KEY in ad_new.obsm:        emb_adv = np.asarray(ad_new.obsm[ADV_KEY], dtype=np.float32)        sil_b_adv = silhouette_score(emb_adv, batch) if mb else np.nan        sil_d_adv = silhouette_score(emb_adv, diag) if md_ else np.nan    else:        sil_b_adv = sil_d_adv = np.nan    rows_cmp.append({        "Cohort": cohort_name,        "Sil_Batch_Raw": round(sil_b_raw, 4) if not np.isnan(sil_b_raw) else np.nan,        "Sil_Batch_KL": round(sil_b_old, 4) if not np.isnan(sil_b_old) else np.nan,        "Sil_Batch_Adv": round(sil_b_adv, 4) if not np.isnan(sil_b_adv) else np.nan,        "Sil_Diag_Raw": round(sil_d_raw, 4) if not np.isnan(sil_d_raw) else np.nan,        "Sil_Diag_KL": round(sil_d_old, 4) if not np.isnan(sil_d_old) else np.nan,        "Sil_Diag_Adv": round(sil_d_adv, 4) if not np.isnan(sil_d_adv) else np.nan,    })    del ad_old, ad_newif rows_cmp:    df_cmp = pd.DataFrame(rows_cmp)    display(df_cmp)    print("\nSil_Batch: ниже = лучше (батчи смешаны)")    print("Sil_Diag:  выше = лучше (биология сохранена)")    print("\nИдеал: Adv имеет Sil_Batch < KL И Sil_Diag > KL")else:    logger.warning("No cohorts found in both processed/ and processed_adv/")